# 🔍 Project 18: Data Quality Engineering (Fundamentals)
### Enterprise-grade Data Quality validation, profiling, alerting, and reporting.

Welcome to this interactive notebook demonstrating the fundamentals of **Data Quality Engineering**. In production data engineering pipelines, maintaining data quality is crucial to prevent downstream analytics and ML models from failing or receiving corrupted inputs.

In this notebook, we will cover:
1. **Rule-Based Validation**: Code-driven and YAML configuration-driven validation.
2. **Data & Schema Profiling**: Statistical profiling and schema drift detection.
3. **Data Partitioning**: Splitting data into clean and quarantine zones.
4. **Historical Monitoring & Alerting**: SQLite database persistence and Slack/Email alert dispatching.
5. **Great Expectations Integration**: Generating declarative validation suites.
6. **HTML Quality Dashboard**: Building standalone visual validation reports.

---

### Prerequisites
Make sure you have installed the required dependencies by running:
```bash
pip install pandas pyyaml scipy great-expectations jinja2 sqlalchemy
```

In [ ]:
import os
import sys
import pandas as pd
import yaml

# Add src to Python path for import compatibility inside notebook
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.dq_framework.quality_rule import Severity, Dimension
from src.dq_framework.quality_engine import QualityEngine
from src.rules.completeness_rules import NullCheckRule, RowCountRule, RequiredColumnsRule
from src.rules.uniqueness_rules import PrimaryKeyUniquenessRule, DuplicateRowsRule
from src.rules.validity_rules import EmailFormatRule, NumericRangeRule, ValueSetRule
from src.profiling.data_profiler import DataProfiler
from src.profiling.schema_profiler import SchemaProfiler
from src.profiling.statistical_profiler import StatisticalProfiler
from src.monitoring.dq_metrics_tracker import DQMetricsTracker
from src.monitoring.dq_alerting import DQAlertDispatcher
from src.monitoring.dq_dashboard import DQDashboard
from src.great_expectations.ge_suite_builder import GESuiteBuilder
from src.great_expectations.ge_runner import GERunner

print("Imports completed successfully!")

In [ ]:
# Load the clean and dirty customer datasets
customers_clean = pd.read_csv("../sample-data/customers_clean.csv")
customers_dirty = pd.read_csv("../sample-data/customers_dirty.csv")

print("--- Clean Customers Sample ---")
print(customers_clean.head(3))
print("\n--- Dirty Customers Sample ---")
print(customers_dirty.head(5))

## 🚦 Step 1: Code-Driven Data Quality Validation

Let's programmatically define a suite of rules to evaluate against our dirty customer dataset. We will use the custom validation rule classes we created:
- `NullCheckRule` to ensure required columns are not empty.
- `PrimaryKeyUniquenessRule` to ensure the unique key constraint.
- `EmailFormatRule` to validate email formats.
- `RowCountRule` to ensure volume requirements are met.

If any `CRITICAL` or `ERROR` rules fail, the engine will mark the report status as `FAILED`. We can then partition our data into **clean (passed)** and **quarantine (failed)** records.

In [ ]:
# 1. Instantiate the individual rules
rules = [
    NullCheckRule(columns=["customer_id", "last_name"], severity=Severity.CRITICAL, description="customer_id and last_name must not be null"),
    PrimaryKeyUniquenessRule(pk_column="customer_id", severity=Severity.ERROR, description="customer_id must be unique"),
    EmailFormatRule(column="email", severity=Severity.WARNING, description="email should match RFC format"),
    RowCountRule(min_rows=5, max_rows=100, severity=Severity.INFO)
]

# 2. Initialize engine and run validation
engine = QualityEngine(rules=rules, dataset_name="customers_eval")
report = engine.run(customers_dirty)

# 3. Print validation report
report.print_summary()

In [ ]:
# Partition data into clean and quarantine DataFrames based on the report
clean_df, quarantine_df = engine.partition_data(customers_dirty, report)

print(f"Total Rows: {len(customers_dirty)}")
print(f"Passed Rows (Clean): {len(clean_df)}")
print(f"Quarantined Rows (Dirty): {len(quarantine_df)}")

print("\n--- Quarantined Records ---")
print(quarantine_df[["customer_id", "last_name", "email"]])

## ⚙️ Step 2: Configuration-Driven Validation

Instead of hardcoding rules in python, modern data engineering platforms store rules in metadata databases or declarative YAML configuration files. Let's read `configs/dq_config.yaml` to dynamically parse and execute the rules configured for the `customers` and `orders` datasets.

In [ ]:
# Load YAML configuration
with open("../configs/dq_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Instantiate engine with config
config_engine = QualityEngine(config=config)

# Validate the customer dataset
clean_customers, quarantine_customers, cust_report = config_engine.validate_dataset(
    table_name="customers", 
    df=customers_dirty
)

print(f"Customer Validation Completed: Status={cust_report.status}")
print(f"Clean Count: {len(clean_customers)} | Quarantine Count: {len(quarantine_customers)}")
cust_report.print_summary()

## 🔬 Step 3: Schema and Data Profiling

Understanding our dataset schema and distributions is the first step in ensuring quality.
- `DataProfiler` helps us scan column statistics, null counts, and distinct value distributions.
- `SchemaProfiler` detects **schema drift** by comparing the dataset columns and types against a configured baseline. Let's see if our dirty customers schema differs from the baseline.

In [ ]:
# 1. Profile customer data
profiler = DataProfiler(dataset_name="customers")
profile_res = profiler.profile(customers_dirty)
print(f"Total Rows: {profile_res['total_rows']}")
print(f"Columns Inferred: {list(profile_res['columns'].keys())}")
print(f"Email Uniqueness Profile: {profile_res['columns']['email']}")

# 2. Setup baseline schema and detect drift
baseline = {
    "customer_id": "object",
    "first_name": "object",
    "last_name": "object",
    "email": "object",
    "phone": "object",
    "signup_date": "object",
    "country": "object",
    "status": "object",
}
schema_profiler = SchemaProfiler(baseline_schema=baseline)
drift_res = schema_profiler.detect_drift(customers_dirty)

print("\n--- Schema Drift Report ---")
print(f"Has Drift: {drift_res['has_drift']}")
print(f"Added Columns: {drift_res['added_columns']}")
print(f"Missing Columns: {drift_res['missing_columns']}")
print(f"Type Mismatches: {drift_res['type_mismatches']}")

## 📊 Step 4: Time-Series Metrics Tracking and Alerting

Historical metric tracking allows data engineers to detect regressions and trends in data quality. 
- `DQMetricsTracker` records quality reports into a SQLite database.
- `DQAlertDispatcher` generates and sends Slack/Email alerts if critical rules fail. Let's record our run and display the historical trend.

In [ ]:
# Initialize SQLite-backed metrics tracker
tracker = DQMetricsTracker(db_path="dq_metrics.db")

# Save cust_report to database
run_id = tracker.record_report(cust_report)
print(f"Recorded report in DB with run_id: {run_id}")

# Fetch execution history
history_df = tracker.get_dataset_history("customers", limit=5)
print("\n--- Historical Runs ---")
print(history_df)

# Dispatch alert
dispatcher = DQAlertDispatcher(alert_config=config.get("monitoring"))
alert_status = dispatcher.dispatch(cust_report)
print(f"\nAlert dispatch statuses: {alert_status}")

## 🔗 Step 5: Great Expectations Integration

Great Expectations (GE) is an industry-standard open-source data quality tool. Our framework includes `GESuiteBuilder` and `GERunner` to map our native python rules into a GE ExpectationSuite structure, validate the dataset using GE, and return a native `QualityReport`. Let's build and run a suite using the Great Expectations emulation engine!

In [ ]:
# 1. Build expectations suite dictionary dynamically from rules
builder = GESuiteBuilder(suite_name="customer_ge_suite")
ge_suite = builder.build_from_rules(rules)
print("--- Dynamic GE Expectation Suite Definition ---")
print(f"Suite Name: {ge_suite['expectation_suite_name']}")
print(f"Expectations Count: {len(ge_suite['expectations'])}")

# 2. Run the suite against the dataset using GERunner
runner = GERunner(dataset_name="customers_ge_run")
ge_report = runner.run_suite(customers_dirty, ge_suite)

# 3. Print report status
print(f"\nGE Validation Completed: Status={ge_report.status}")
ge_report.print_summary()

## 📈 Step 6: HTML Quality Dashboard

A standalone HTML dashboard provides a visual report card that can be easily published to S3, Jenkins, Airflow, or shared with business stakeholders. We will use the `DQDashboard` module to compile our validation report into an HTML file.

In [ ]:
# 1. Initialize the dashboard generator
dashboard = DQDashboard(template_title="Data Engineering Diaries Quality Portal")

# 2. Compile report into HTML dashboard
dashboard_file = "customer_quality_dashboard.html"
abs_path = dashboard.generate(cust_report, output_path=dashboard_file)

print("--- Quality Dashboard Generated ---")
print(f"File Saved: {dashboard_file}")
print(f"Absolute Path: {abs_path}")